In [1]:
import geopandas as gpd
import pandas as pd
import shapely
import os
from tqdm import tqdm


In [2]:
exported_error = gpd.read_file(
    os.path.join(
        'data',
        'pontos_buff_v4.0.geojson'
    )
)

In [3]:
#copia
drenageo = gpd.read_file(
    os.path.join(
        'data',
        'drenagem.zip'
    )
)

In [4]:
#copia
## Como as correntes estimadas tem a id como int, vou transformar a id do drenageo em int tbm
drenageo['cd_identif'].astype('int', copy=False)
## Conferir se todos os 'cd_tipo_cu' sejam do mesmo tipo
drenageo['cd_tipo_cu'].dtype

dtype('float64')

In [5]:
#copia
drenageo.sample(10)
#* 11: trecho em estado natural
#* 12: lago ou reservatório
#* 10: trecho fechado
#* 9: trecho a céu aberto

cus_to_keep = [9.0, 11.0]
colors_dictionarie= {
    9.0 : 'turquoise',
    11.0 : 'aquamarine',
    10.0 : 'pink',
    12.0 : 'pink',
}

drenageo['colors'] = drenageo['cd_tipo_cu'].map(colors_dictionarie)

In [6]:
#copia
drena= drenageo[[
    'cd_identif', 
    'cd_tipo_ac', 
    'cd_tipo_cu', 
    'nm_acident', 
    'geometry',
    'nm_bairro'
]]

# Funções (also copia)

In [7]:
# Função drop_intersec

def drop_intersec(pontos, linhas):
    intersecs_bool=[]

    for i, row in pontos.iterrows():
        outras_geoms = pontos.loc[pontos.index!=i]
        intersecs_bool = row.geometry.intersects(outras_geoms.geometry)
        if len(intersecs_bool.loc[intersecs_bool==True])<1:
            pontos.loc[pontos.index==i, 'intersec_bool'] = False
        else:
            pontos.loc[pontos.index==i, 'intersec_bool'] = True
    
    for i, row in pontos.loc[pontos['intersec_bool']==False].iterrows():
        outras_linhas = linhas.loc[linhas['cd_identif']!=row['cd_identif']]
        intersecs_bool = row.geometry.intersects(outras_linhas.geometry)
        if len(intersecs_bool.loc[intersecs_bool==True])<1:
            pontos.loc[pontos.index==i, 'intersec_bool']=False
        else:
            pontos.loc[pontos.index==i, 'intersec_bool'] = True

    pontos = pontos.loc[pontos['intersec_bool']==False]
    

    return pontos

In [8]:
#funções dos pontos

def get_pontos(gdf):
    pontos = gdf.copy()
    pontos_0 = gdf.copy()
    pontos_1 = gdf.copy()

    pontos_0['geometry']=shapely.get_point(gdf.geometry, 0)
    pontos_0['cd_point'] = pontos_0['cd_identif'].astype(int).astype(str)+".0"
    pontos_1['geometry'] = shapely.get_point(gdf.geometry, -1)
    pontos_1['cd_point'] = pontos_1['cd_identif'].astype(int).astype(str)+".1"
    pontos = pd.concat([pontos_0, pontos_1], ignore_index=True)
    
    return pontos
    
def get_pontos_buff(
    #gdf,
    pontos,
    buffer
):
    #pontos = get_pontos(gdf)
    pontos_buff = pontos.copy()
    pontos_buff['geometry'] = pontos['geometry'].buffer(buffer)
    return pontos_buff


def intersec_buffed(gdf, pontos, buffer):
    buffed=get_pontos_buff(pontos, buffer)
    final = drop_intersec(pontos=buffed, linhas=gdf)
    return final


#esta aqui é nova:
def drop_subterraneos(pontos_buff):
    pontos_buff['cd_tipo_cu'].astype(dtype='float', copy=False)
    pontos_buff= pontos_buff.loc[pontos_buff['cd_tipo_cu'].isin(cus_to_keep)]
    return pontos_buff

In [9]:
teste_v4 = drenageo.loc[drenageo['cd_identif']==10226.0]

In [10]:
bairros =drenageo.loc[drenageo['nm_acident'].str.contains('GOLFE'), 'nm_bairro'].unique()
recort = drena.loc[drena['nm_bairro'].isin(bairros)]

In [11]:
recort_pontos = get_pontos(recort)
buffer10 = intersec_buffed(recort, recort_pontos, 10)
buffer10drop = drop_subterraneos(buffer10)

buffer9 = intersec_buffed(recort, recort_pontos, 9)
buffer9drop = drop_subterraneos(buffer9)

buffer8=intersec_buffed(recort, recort_pontos,8)
buffer8drop = drop_subterraneos(buffer8)

buffer7=intersec_buffed(recort, recort_pontos,7)
buffer7drop = drop_subterraneos(buffer7)

buffer6=intersec_buffed(recort, recort_pontos,6)
buffer6drop = drop_subterraneos(buffer6)

buffer5 = intersec_buffed(recort, recort_pontos, 5)
buffer5drop = drop_subterraneos(buffer5)

In [12]:
print(
    f'Shape das linhas: {recort.shape}\n'
    + f'O dobro disso é: {recort.shape[0]*2}'
)
print(
    f'Shape dos pontos: {recort_pontos.shape}'
)
print(f'''
PONTOS:
    10: {buffer10.shape}
    9: {buffer9.shape}
    8: {buffer8.shape}
    7: {buffer7.shape}
    6: {buffer6.shape}
    5: {buffer5.shape}


''')

print(f'''
PONTOS CUS TO KEEP:
    10: {buffer10drop.shape}
    9: {buffer9drop.shape}
    8: {buffer8drop.shape}
    7: {buffer7drop.shape}
    6: {buffer6drop.shape}
    5: {buffer5drop.shape}


''')

Shape das linhas: (9120, 6)
O dobro disso é: 18240
Shape dos pontos: (18240, 7)

PONTOS:
    10: (4375, 8)
    9: (4413, 8)
    8: (4448, 8)
    7: (4489, 8)
    6: (4527, 8)
    5: (4571, 8)




PONTOS CUS TO KEEP:
    10: (4134, 8)
    9: (4164, 8)
    8: (4196, 8)
    7: (4229, 8)
    6: (4257, 8)
    5: (4289, 8)





In [13]:
recort_pontos.sample()

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry,nm_bairro,cd_point
17436,23967.0,ND,11.0,SD,POINT (327038.659 7406218.754),S/B,23967.1


In [14]:
pontos10 = recort_pontos.loc[recort_pontos['cd_point'].isin(buffer10drop['cd_point'])]
pontos9= recort_pontos.loc[recort_pontos['cd_point'].isin(buffer9drop['cd_point'])]
pontos8= recort_pontos.loc[recort_pontos['cd_point'].isin(buffer8drop['cd_point'])]
pontos7= recort_pontos.loc[recort_pontos['cd_point'].isin(buffer7drop['cd_point'])]
pontos6= recort_pontos.loc[recort_pontos['cd_point'].isin(buffer6drop['cd_point'])]
pontos5= recort_pontos.loc[recort_pontos['cd_point'].isin(buffer5drop['cd_point'])]

# Calcular faltantes

falt9 = pontos10.loc[~pontos10['cd_identif'].isin(pontos9)]
falt8 = pontos10.loc[~pontos10['cd_identif'].isin(pontos8)]
falt7 = pontos10.loc[~pontos10['cd_identif'].isin(pontos7)]
falt6 = pontos10.loc[~pontos10['cd_identif'].isin(pontos6)]
falt5 = pontos10.loc[~pontos10['cd_identif'].isin(pontos5)]

# Visualizar

m= recort.explore(color='pink')

#recort.loc[recort['cd_tipo_cu'].isin(cus_to_keep)].explore(m=m,color='blue')

teste_v4.explore(m=m, color='red')

#exported_error.explore(m=m, color="green")


pontos5.explore( #com 6m já passa a dar erro # mas só pega com 5m
    m=m,
    color='purple'
)
pontos10.explore(m=m, color='green')